# 🇮🇳 AI-Powered Smart Fuel Management System
**4 ML Models** | XGBoost · Isolation Forest · Prophet · Efficiency Ranking

> Data: India 2024–2025 | Currency: ₹ INR

In [ ]:
# ─── Step 0 : Install dependencies ───────────────────────────────────────────
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost prophet streamlit plotly joblib

In [ ]:
# ─── Step 1 : Imports ─────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics         import precision_score, recall_score, classification_report
from sklearn.ensemble        import IsolationForest
from sklearn.preprocessing   import StandardScaler
from xgboost                 import XGBRegressor
try:
    from prophet import Prophet
except:
    from fbprophet import Prophet

print('✅ All libraries imported successfully')

## 📦 Step 2 – Generate Indian Fuel Dataset (2024–2025)

In [ ]:
np.random.seed(42)

CITIES_STATES = {
    'Delhi':'Delhi','Mumbai':'Maharashtra','Bengaluru':'Karnataka',
    'Chennai':'Tamil Nadu','Hyderabad':'Telangana','Kolkata':'West Bengal',
    'Pune':'Maharashtra','Ahmedabad':'Gujarat'
}
VEHICLE_TYPES  = ['Truck','Bus','Car','Van','SUV','Mini-Truck']
FUEL_TYPES     = ['Petrol','Diesel','CNG']
DEPARTMENTS    = ['Logistics','Sales','Operations','Maintenance','Administration']
ROUTE_TYPES    = ['Urban','Highway','Mixed']
FUEL_PRICES    = {'Petrol':(94,105),'Diesel':(85,95),'CNG':(75,85)}
MILEAGE_RANGE  = {'Truck':(5,12),'Bus':(4,8),'Car':(12,22),'Van':(10,16),'SUV':(8,14),'Mini-Truck':(8,14)}

dates  = pd.date_range('2024-01-01','2025-12-31',freq='D')
cities = list(CITIES_STATES.keys())
rows   = []

for i in range(1,5001):
    city  = np.random.choice(cities)
    vtype = np.random.choice(VEHICLE_TYPES)
    ftype = np.random.choice(FUEL_TYPES, p=[0.35,0.50,0.15])
    route = np.random.choice(ROUTE_TYPES)
    date  = np.random.choice(dates)
    price = round(np.random.uniform(*FUEL_PRICES[ftype]),2)
    mil   = round(np.random.uniform(*MILEAGE_RANGE[vtype]),2)
    base  = 200 if route=='Highway' else (80 if route=='Urban' else 130)
    dist  = round(abs(np.random.normal(base,40)),1)
    qty   = round(dist/mil,2)
    cost  = round(qty*price,2)
    maint = round(abs(np.random.normal(1200,500)),2)
    fraud = 0
    if np.random.rand()<0.05:
        fraud=1
        ft=np.random.choice(['high_bill','low_km_high_fuel'])
        if ft=='high_bill': qty=round(qty*3,2); cost=round(qty*price,2)
        else: dist=round(dist*0.2,1); qty=round(qty*2,2); cost=round(qty*price,2); mil=round(dist/max(qty,0.1),2)
    rows.append({
        'Date':date.strftime('%Y-%m-%d'),'Vehicle_ID':f'VH{1000+(i%200):04d}',
        'Vehicle_Type':vtype,'Driver_ID':f'DR{500+(i%100):04d}',
        'City':city,'State':CITIES_STATES[city],'Fuel_Type':ftype,
        'Fuel_Quantity_Liters':qty,'Fuel_Price_Per_Liter_INR':price,
        'Total_Fuel_Cost_INR':cost,'Distance_KM':dist,'Mileage_KMPL':mil,
        'Maintenance_Cost_INR':maint,'Department':np.random.choice(DEPARTMENTS),
        'Route_Type':route,'Is_Fraud':fraud
    })

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'])
df.to_csv('fuel.csv',index=False)
print(f'✅ Dataset created: {df.shape}')
df.head(3)

## 🧹 Step 3 – Data Cleaning & EDA

In [ ]:
# ── Cleaning ──────────────────────────────────────────────────────────────────
print('Missing values:'); print(df.isnull().sum())
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
# Remove impossible mileage values
df = df[(df['Mileage_KMPL']>0) & (df['Distance_KM']>0) & (df['Fuel_Quantity_Liters']>0)]
print(f'\nCleaned dataset shape: {df.shape}')

In [ ]:
# ── EDA Charts ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(18,10))
fig.suptitle('🇮🇳 India Fleet Fuel Analysis – EDA',fontsize=15,fontweight='bold')

# 1. Monthly fuel cost
monthly = df.set_index('Date').resample('ME')['Total_Fuel_Cost_INR'].sum()/1e6
monthly.plot(ax=axes[0,0],marker='o',color='steelblue')
axes[0,0].set_title('Monthly Fuel Cost (₹ Millions)'); axes[0,0].set_ylabel('₹ M')

# 2. City fuel spending
city_cost = df.groupby('City')['Total_Fuel_Cost_INR'].sum().sort_values(ascending=False)/1e6
city_cost.plot(kind='bar',ax=axes[0,1],color='coral')
axes[0,1].set_title('City-wise Fuel Spending (₹ M)'); axes[0,1].tick_params(axis='x',rotation=45)

# 3. Vehicle type vs fuel
vt = df.groupby('Vehicle_Type')['Fuel_Quantity_Liters'].mean().sort_values()
vt.plot(kind='barh',ax=axes[0,2],color='mediumseagreen')
axes[0,2].set_title('Avg Fuel per Trip by Vehicle Type')

# 4. Mileage distribution
axes[1,0].hist(df['Mileage_KMPL'],bins=40,color='orchid',edgecolor='white')
axes[1,0].set_title('Mileage (KMPL) Distribution')

# 5. Fuel type pie
ft_counts = df['Fuel_Type'].value_counts()
axes[1,1].pie(ft_counts,labels=ft_counts.index,autopct='%1.1f%%',
               colors=['#FF6B6B','#4ECDC4','#45B7D1'])
axes[1,1].set_title('Fuel Type Distribution')

# 6. Correlation heatmap
num_cols=['Fuel_Quantity_Liters','Total_Fuel_Cost_INR','Distance_KM','Mileage_KMPL','Maintenance_Cost_INR']
sns.heatmap(df[num_cols].corr(),annot=True,fmt='.2f',cmap='RdBu',ax=axes[1,2])
axes[1,2].set_title('Correlation Heatmap')

plt.tight_layout(); plt.savefig('eda_charts.png',dpi=150); plt.show()
print('✅ EDA charts saved')

## 🤖 MODEL 1 – Fuel Consumption Prediction (XGBoost)

In [ ]:
df2 = df.copy()
df2['Month']=df2['Date'].dt.month; df2['DayOfWeek']=df2['Date'].dt.dayofweek; df2['Quarter']=df2['Date'].dt.quarter

cat_cols=['Vehicle_Type','Fuel_Type','City','State','Department','Route_Type']
le_dict={}
for col in cat_cols:
    le=LabelEncoder(); df2[col]=le.fit_transform(df2[col].astype(str)); le_dict[col]=le

features=['Month','DayOfWeek','Quarter','Vehicle_Type','Fuel_Type','City','State',
           'Fuel_Price_Per_Liter_INR','Distance_KM','Mileage_KMPL','Maintenance_Cost_INR','Department','Route_Type']
X=df2[features]; y=df2['Fuel_Quantity_Liters']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

xgb=XGBRegressor(n_estimators=300,learning_rate=0.05,max_depth=6,subsample=0.8,
                  colsample_bytree=0.8,random_state=42,verbosity=0)
xgb.fit(X_train,y_train,eval_set=[(X_test,y_test)],verbose=False)

y_pred=xgb.predict(X_test)
rmse=np.sqrt(mean_squared_error(y_test,y_pred))
print(f'RMSE : {rmse:.4f} litres')
print(f'MAE  : {mean_absolute_error(y_test,y_pred):.4f} litres')
print(f'R²   : {r2_score(y_test,y_pred):.4f}')

fi=pd.Series(xgb.feature_importances_,index=features).sort_values(ascending=False)
fi.head(10).plot(kind='barh',figsize=(8,4),color='steelblue')
plt.title('Feature Importances – Fuel Prediction'); plt.tight_layout(); plt.show()

## 🚨 MODEL 2 – Fraud Detection (Isolation Forest)

In [ ]:
df_fr=df.copy()
df_fr['Cost_Per_KM']=df_fr['Total_Fuel_Cost_INR']/df_fr['Distance_KM'].replace(0,np.nan)
df_fr['Fuel_Per_KM']=df_fr['Fuel_Quantity_Liters']/df_fr['Distance_KM'].replace(0,np.nan)
df_fr['Expected_Fuel']=df_fr['Distance_KM']/df_fr['Mileage_KMPL'].replace(0,np.nan)
df_fr['Fuel_Excess']=df_fr['Fuel_Quantity_Liters']-df_fr['Expected_Fuel']
df_fr['Price_Dev']=df_fr['Fuel_Price_Per_Liter_INR']-df_fr.groupby('Fuel_Type')['Fuel_Price_Per_Liter_INR'].transform('mean')

feat_fr=['Fuel_Quantity_Liters','Total_Fuel_Cost_INR','Distance_KM',
          'Mileage_KMPL','Cost_Per_KM','Fuel_Per_KM','Fuel_Excess','Price_Dev']
Xf=df_fr[feat_fr].fillna(0)
sc=StandardScaler(); Xfs=sc.fit_transform(Xf)

iso=IsolationForest(n_estimators=200,contamination=0.05,random_state=42)
iso.fit(Xfs)
df_fr['Anomaly']=(iso.predict(Xfs)==-1).astype(int)
df_fr['Anomaly_Score']=-iso.score_samples(Xfs)

print(f'Fraud Alerts: {df_fr["Anomaly"].sum()}')
print(classification_report(df_fr['Is_Fraud'],df_fr['Anomaly'],target_names=['Normal','Fraud']))

fraud_df=df_fr[df_fr['Anomaly']==1].copy()
plt.figure(figsize=(9,5))
plt.scatter(df_fr[df_fr['Anomaly']==0]['Distance_KM'],df_fr[df_fr['Anomaly']==0]['Total_Fuel_Cost_INR'],
             c='steelblue',alpha=0.3,s=12,label='Normal')
plt.scatter(fraud_df['Distance_KM'],fraud_df['Total_Fuel_Cost_INR'],
             c='crimson',alpha=0.8,s=40,label='Fraud',zorder=3)
plt.xlabel('Distance (KM)'); plt.ylabel('Total Fuel Cost (₹)'); plt.title('Fraud Detection')
plt.legend(); plt.tight_layout(); plt.show()

## 📉 MODEL 3 – Cost Forecasting (Prophet)

In [ ]:
daily=df.groupby('Date')['Total_Fuel_Cost_INR'].sum().reset_index()
daily.columns=['ds','y']; daily=daily.sort_values('ds')

m=Prophet(yearly_seasonality=True,weekly_seasonality=True,seasonality_mode='multiplicative')
m.add_seasonality(name='monthly',period=30.5,fourier_order=5)
m.fit(daily)

future=m.make_future_dataframe(periods=365,freq='D')
fc=m.predict(future)

# Monthly aggregation
fc['Month']=fc['ds'].dt.to_period('M')
monthly_fc=fc.groupby('Month').agg(Forecast=('yhat','sum'),
                                     Lower=('yhat_lower','sum'),
                                     Upper=('yhat_upper','sum')).reset_index()
monthly_fc['Month']=monthly_fc['Month'].astype(str)

last_m=daily['ds'].max().to_period('M').strftime('%Y-%m')
future_m=monthly_fc[monthly_fc['Month']>last_m].head(12)
print('\n12-Month Forecast (₹ INR):')
print(future_m.to_string(index=False))

fig=m.plot(fc); plt.title('Fuel Cost Forecast – Prophet'); plt.tight_layout(); plt.show()
m.plot_components(fc); plt.tight_layout(); plt.show()

## 🏆 MODEL 4 – Vehicle Efficiency Ranking

In [ ]:
agg=df.groupby('Vehicle_ID').agg(
    Total_Distance_KM=('Distance_KM','sum'),
    Total_Fuel_Liters=('Fuel_Quantity_Liters','sum'),
    Total_Cost_INR=('Total_Fuel_Cost_INR','sum'),
    Total_Maint_INR=('Maintenance_Cost_INR','sum'),
    Avg_Mileage_KMPL=('Mileage_KMPL','mean'),
    Trips=('Distance_KM','count')
).reset_index()

agg['Cost_Per_KM']=agg['Total_Cost_INR']/agg['Total_Distance_KM'].replace(0,np.nan)

def norm(s): return (s-s.min())/(s.max()-s.min()+1e-9)
agg['Score']=(0.35*norm(agg['Avg_Mileage_KMPL'])+
               0.35*(1-norm(agg['Cost_Per_KM']))+
               0.15*(1-norm(agg['Total_Maint_INR']))+
               0.15*norm(agg['Total_Distance_KM'])).round(4)

agg=agg.sort_values('Score',ascending=False).reset_index(drop=True)
agg['Rank']=agg.index+1

print('🏆 TOP 10'); print(agg[['Rank','Vehicle_ID','Avg_Mileage_KMPL','Cost_Per_KM','Score']].head(10).to_string(index=False))
print('\n⚠️  BOTTOM 10'); print(agg[['Rank','Vehicle_ID','Avg_Mileage_KMPL','Cost_Per_KM','Score']].tail(10).to_string(index=False))

fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=agg.head(10),x='Score',y='Vehicle_ID',palette='Greens_r',ax=axes[0])
axes[0].set_title('Top 10 Efficient Vehicles')
sns.barplot(data=agg.tail(10).sort_values('Score'),x='Score',y='Vehicle_ID',palette='Reds',ax=axes[1])
axes[1].set_title('Bottom 10 Inefficient Vehicles')
plt.tight_layout(); plt.show()

## ✅ All 4 Models Complete!

| Model | Algorithm | Task |
|-------|-----------|------|
| 1 | XGBoost Regressor | Fuel Consumption Prediction |
| 2 | Isolation Forest | Fraud Detection |
| 3 | Prophet | Cost Forecasting |
| 4 | Weighted Scoring | Vehicle Efficiency Ranking |

> To run the Streamlit dashboard: `streamlit run app.py`